# Actividad Predicción de precios de casas en California con PyTorch
Axel Daniel Nuño | T035859

En esta actividad se continúa con el notebook anterior, pero ahora usando una red neuronal en PyTorch. Se carga el dataset de California Housing y se divide en train, val y test (80/10/10). El preprocesamiento se integra dentro del modelo para evitar data leakage, dejando Latitude y Longitude sin cambios y aplicando limpieza de outliers y scaling al resto usando solo datos de entrenamiento. Finalmente, se entrenan tres modelos con distintos hiperparámetros y se selecciona el mejor con base en validación.

In [1]:
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.2, random_state=42
)

x_val, x_test, y_val, y_test = train_test_split(
    x_temp, y_temp, test_size=0.5, random_state=42
)

feature_names = list(x.columns)
geo_features = ["Latitude", "Longitude"]
processed_features = [c for c in feature_names if c not in geo_features]

x_train_tensor = torch.tensor(x_train.values, dtype=torch.float32)
x_val_tensor = torch.tensor(x_val.values, dtype=torch.float32)
x_test_tensor = torch.tensor(x_test.values, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

print("Train:", x_train.shape, y_train.shape)
print("Val:", x_val.shape, y_val.shape)
print("Test:", x_test.shape, y_test.shape)

Train: (16512, 8) (16512,)
Val: (2064, 8) (2064,)
Test: (2064, 8) (2064,)


## Preprocesamiento dentro del modelo

Aquí el preprocesamiento ya se mete directo dentro del modelo con PyTorch, para no depender de pasos externos y evitar data leakage. A todas las variables menos `Latitude` y `Longitude` se les quitan outliers usando IQR y luego se les aplica standard scaling. Las coordenadas se dejan tal cual. Todo se ajusta solo con el set de entrenamiento.

In [2]:
class HousingPreprocessor(nn.Module):
    def __init__(self, feature_names, geo_features):
        super().__init__()
        self.feature_names = feature_names
        self.geo_features = geo_features
        self.proc_indices = [feature_names.index(c) for c in feature_names if c not in geo_features]
        self.register_buffer("lower_bounds", torch.zeros(len(self.proc_indices)))
        self.register_buffer("upper_bounds", torch.zeros(len(self.proc_indices)))
        self.register_buffer("means", torch.zeros(len(self.proc_indices)))
        self.register_buffer("stds", torch.ones(len(self.proc_indices)))

    def fit(self, x_train):
        x_proc = x_train[:, self.proc_indices]
        q1 = torch.quantile(x_proc, 0.25, dim=0)
        q3 = torch.quantile(x_proc, 0.75, dim=0)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        clipped = torch.clamp(x_proc, min=lower, max=upper)
        means = clipped.mean(dim=0)
        stds = clipped.std(dim=0, unbiased=False)
        stds = torch.where(stds == 0, torch.ones_like(stds), stds)
        self.lower_bounds.copy_(lower)
        self.upper_bounds.copy_(upper)
        self.means.copy_(means)
        self.stds.copy_(stds)

    def forward(self, x):
        x_out = x.clone()
        x_proc = x_out[:, self.proc_indices]
        x_proc = torch.maximum(x_proc, self.lower_bounds.unsqueeze(0))
        x_proc = torch.minimum(x_proc, self.upper_bounds.unsqueeze(0))
        x_proc = (x_proc - self.means.unsqueeze(0)) / self.stds.unsqueeze(0)
        x_out[:, self.proc_indices] = x_proc
        return x_out

## Definición del modelo de red neuronal

Aquí se define la red neuronal de regresión, integrando el preprocesamiento directamente dentro del modelo. Así, los datos primero pasan por la limpieza y el scaling, y luego por las capas densas para hacer la predicción. Esto deja todo el flujo en un solo modelo y facilita probar diferentes configuraciones.

In [3]:
class HousingRegressor(nn.Module):
    def __init__(self, feature_names, geo_features, hidden_layers, dropout=0.0):
        super().__init__()
        self.preprocessor = HousingPreprocessor(feature_names, geo_features)
        layers = []
        input_dim = len(feature_names)
        dims = [input_dim] + list(hidden_layers)

        for i in range(len(dims) - 1):
            layers.append(nn.Linear(dims[i], dims[i + 1]))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))

        layers.append(nn.Linear(dims[-1], 1))
        self.network = nn.Sequential(*layers)

    def fit_preprocessor(self, x_train):
        self.preprocessor.fit(x_train)

    def forward(self, x):
        x = self.preprocessor(x)
        return self.network(x)

## Preparación de DataLoaders

Aquí se preparan los tensores y los `DataLoader` para train, val y test. Con esto ya queda todo listo para entrenar y probar diferentes modelos sin tener que rearmar el flujo cada vez.

In [4]:
batch_size = 256

train_dataset = TensorDataset(x_train_tensor, y_train_tensor)
val_dataset = TensorDataset(x_val_tensor, y_val_tensor)
test_dataset = TensorDataset(x_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

## Funciones de entrenamiento y evaluación

Aquí se definen las funciones para entrenar los modelos y evaluar su desempeño. Se usan métricas como MAE, MSE, RMSE y R² para comparar resultados. El mejor modelo se elige con validación y el test se deja solo para la evaluación final.

In [5]:
def regression_metrics(y_true, y_pred):
    y_true = y_true.view(-1)
    y_pred = y_pred.view(-1)
    mae = torch.mean(torch.abs(y_true - y_pred)).item()
    mse = torch.mean((y_true - y_pred) ** 2).item()
    rmse = mse ** 0.5
    ss_res = torch.sum((y_true - y_pred) ** 2)
    ss_tot = torch.sum((y_true - torch.mean(y_true)) ** 2)
    r2 = (1 - ss_res / ss_tot).item()
    return {"MAE": mae, "MSE": mse, "RMSE": rmse, "R2": r2}

def evaluate_model(model, loader, loss_fn):
    model.eval()
    losses = []
    y_true_all = []
    y_pred_all = []

    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            preds = model(xb)
            loss = loss_fn(preds, yb)
            losses.append(loss.item())
            y_true_all.append(yb.cpu())
            y_pred_all.append(preds.cpu())

    y_true_all = torch.cat(y_true_all, dim=0)
    y_pred_all = torch.cat(y_pred_all, dim=0)
    metrics = regression_metrics(y_true_all, y_pred_all)
    metrics["loss"] = float(np.mean(losses))
    return metrics

def train_model(model, train_loader, val_loader, x_train_tensor, epochs=100, lr=1e-3, weight_decay=0.0):
    model = model.to(device)
    model.fit_preprocessor(x_train_tensor.to(device))
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    loss_fn = nn.MSELoss()

    history = []
    best_state = None
    best_val_rmse = float("inf")

    for epoch in range(epochs):
        model.train()
        train_losses = []

        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            optimizer.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            optimizer.step()

            train_losses.append(loss.item())

        train_metrics = evaluate_model(model, train_loader, loss_fn)
        val_metrics = evaluate_model(model, val_loader, loss_fn)

        history.append({
            "epoch": epoch + 1,
            "train_loss": float(np.mean(train_losses)),
            "train_rmse": train_metrics["RMSE"],
            "val_rmse": val_metrics["RMSE"]
        })

        if val_metrics["RMSE"] < best_val_rmse:
            best_val_rmse = val_metrics["RMSE"]
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history)

## Entrenamiento de tres modelos con diferentes hiperparámetros

Aquí se entrenan tres modelos con distintas configuraciones (capas, neuronas, dropout, etc.) para comparar resultados. La idea es probar varias opciones y quedarse con la que mejor funcione en validación.

In [6]:
configs = [
    {
        "name": "modelo_1",
        "hidden_layers": [64, 32],
        "dropout": 0.0,
        "epochs": 120,
        "lr": 1e-3,
        "weight_decay": 0.0
    },
    {
        "name": "modelo_2",
        "hidden_layers": [128, 64, 32],
        "dropout": 0.1,
        "epochs": 150,
        "lr": 7e-4,
        "weight_decay": 1e-5
    },
    {
        "name": "modelo_3",
        "hidden_layers": [256, 128, 64],
        "dropout": 0.2,
        "epochs": 180,
        "lr": 5e-4,
        "weight_decay": 1e-4
    }
]

trained_models = {}
histories = {}
results = []

for config in configs:
    model = HousingRegressor(
        feature_names=feature_names,
        geo_features=geo_features,
        hidden_layers=config["hidden_layers"],
        dropout=config["dropout"]
    )

    trained_model, history_df = train_model(
        model=model,
        train_loader=train_loader,
        val_loader=val_loader,
        x_train_tensor=x_train_tensor,
        epochs=config["epochs"],
        lr=config["lr"],
        weight_decay=config["weight_decay"]
    )

    loss_fn = nn.MSELoss()
    train_metrics = evaluate_model(trained_model, train_loader, loss_fn)
    val_metrics = evaluate_model(trained_model, val_loader, loss_fn)

    trained_models[config["name"]] = trained_model
    histories[config["name"]] = history_df

    results.append({
        "modelo": config["name"],
        "hidden_layers": str(config["hidden_layers"]),
        "dropout": config["dropout"],
        "epochs": config["epochs"],
        "lr": config["lr"],
        "weight_decay": config["weight_decay"],
        "train_rmse": train_metrics["RMSE"],
        "val_rmse": val_metrics["RMSE"],
        "train_mae": train_metrics["MAE"],
        "val_mae": val_metrics["MAE"],
        "train_r2": train_metrics["R2"],
        "val_r2": val_metrics["R2"]
    })

results_df = pd.DataFrame(results).sort_values("val_rmse").reset_index(drop=True)
results_df

,modelo,hidden_layers,dropout,epochs,lr,weight_decay,train_rmse,val_rmse,train_mae,val_mae,train_r2,val_r2
0,modelo_2,"[128, 64, 32]",0.1,150,0.0007,0.00001,0.622696,0.630187,0.440623,0.438050,0.709937,0.698400
1,modelo_3,"[256, 128, 64]",0.2,180,0.0005,0.00010,0.627768,0.637701,0.449119,0.448302,0.705192,0.691165
2,modelo_1,"[64, 32]",0.0,120,0.0010,0.00000,0.658065,0.679240,0.479872,0.487948,0.676050,0.649621


## Selección del mejor modelo

Aquí se selecciona el mejor modelo usando el RMSE de validación. El set de prueba se deja aparte y solo se usa al final para evaluar el resultado.

In [7]:
best_model_name = results_df.loc[0, "modelo"]
best_model = trained_models[best_model_name]

print("Mejor modelo:", best_model_name)
results_df

Mejor modelo: modelo_2


,modelo,hidden_layers,dropout,epochs,lr,weight_decay,train_rmse,val_rmse,train_mae,val_mae,train_r2,val_r2
0,modelo_2,"[128, 64, 32]",0.1,150,0.0007,0.00001,0.622696,0.630187,0.440623,0.438050,0.709937,0.698400
1,modelo_3,"[256, 128, 64]",0.2,180,0.0005,0.00010,0.627768,0.637701,0.449119,0.448302,0.705192,0.691165
2,modelo_1,"[64, 32]",0.0,120,0.0010,0.00000,0.658065,0.679240,0.479872,0.487948,0.676050,0.649621


## Evaluación final del mejor modelo

Aquí se evalúa el mejor modelo en train, val y test para ver qué tan bien generaliza. Con esto ya se tiene el resultado final de la actividad.

In [8]:
loss_fn = nn.MSELoss()

best_train_metrics = evaluate_model(best_model, train_loader, loss_fn)
best_val_metrics = evaluate_model(best_model, val_loader, loss_fn)
best_test_metrics = evaluate_model(best_model, test_loader, loss_fn)

final_metrics_df = pd.DataFrame({
    "Train": best_train_metrics,
    "Validation": best_val_metrics,
    "Test": best_test_metrics
}).T

final_metrics_df

,MAE,MSE,RMSE,R2,loss
Train,0.440623,0.387750,0.622696,0.709937,0.387109
Validation,0.438050,0.397136,0.630187,0.698400,0.457864
Test,0.448434,0.400210,0.632622,0.692941,0.417014


## Historial de entrenamiento del mejor modelo

Para complementar la evaluación, también se puede revisar cómo evolucionó el entrenamiento del mejor modelo a lo largo de las épocas. Esto ayuda a ver si el comportamiento entre entrenamiento y validación fue estable y también permite detectar si hubo señales de sobreajuste o mejora progresiva.

In [9]:
histories[best_model_name].tail(10)

,epoch,train_loss,train_rmse,val_rmse
140,141,0.412917,0.664356,0.667588
141,142,0.409259,0.635297,0.640193
142,143,0.413179,0.657487,0.659886
143,144,0.413674,0.632356,0.636538
144,145,0.415244,0.641070,0.646228
145,146,0.415998,0.631088,0.634798
146,147,0.410178,0.663214,0.663566
147,148,0.415876,0.637838,0.642269
148,149,0.412319,0.655698,0.658937
149,150,0.416396,0.626647,0.633437


## Conclusión

Con esto se completa la actividad usando PyTorch tanto para el modelo como para el preprocesamiento. Se respetó la división 80/10/10, se evitó data leakage y se probaron tres modelos distintos. Al final se seleccionó el mejor con validación y se evaluó en test.